In [0]:
"""
Spotify MPD - Complete Pipeline for Databricks Community Edition
================================================================
Authors: Diessner, Fiedler, Ryciuk, Leonetti Luparini, De Tuddo

Fixes included:
- row_number() instead of monotonically_increasing_id() (Long overflow fix)
- recommendForAllUsers saved to Parquet before use (Unity Catalog fix)
- No higher-order functions anywhere (Unity Catalog compatible)
- model.write().overwrite().save() to allow re-runs
- ALSModel cache cleared before loading

SETUP:
1. Fill in KAGGLE_USERNAME and KAGGLE_KEY
2. Copy entire script into a single Python cell in Databricks
3. Run
"""

import os
import subprocess
import math
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import FloatType, ArrayType, StructType, StructField, IntegerType
from pyspark.sql.functions import pandas_udf
from pyspark.ml.recommendation import ALS, ALSModel
from pyspark.ml.evaluation import RegressionEvaluator

# ============================================================
# 1. CONFIGURATION
# ============================================================

KAGGLE_USERNAME = ""   # fill in
KAGGLE_KEY      = ""   # fill in

CATALOG = "spotify_project"
SCHEMA  = "mpd"
RAW_VOL = f"/Volumes/{CATALOG}/{SCHEMA}/raw/"
OUT_VOL = f"/Volumes/{CATALOG}/{SCHEMA}/outputs/"

ALS_RANK       = 20      # max feasible on Serverless (rank=50 → 618MB → exceeds 256MB limit)
ALS_MAX_ITER   = 5
ALS_REG_PARAM  = 0.05    # was 0.1 — less regularization improves quality
ALS_ALPHA      = 60.0    # was 40.0 — stronger implicit confidence weighting
ALS_SEED       = 42
K              = 10
CANDIDATE_K    = 100     # candidates passed to re-ranker (was 50)
EVAL_SAMPLE    = 2000    # evaluate on 2K sampled test users (statistically valid, stable)

# Set True to delete old ALS checkpoint and retrain with improved params (~6-8h).
# Set False to reuse existing model and only improve re-ranking (fast).
FORCE_RETRAIN   = False  # reuse trained model

HIGH_PERCENTILE = 0.90
MID_PERCENTILE  = 0.70
LONGTAIL_BOOST  = 1.5    # was 1.3
MIDTAIL_BOOST   = 1.2    # was 1.1

# ============================================================
# 2. SPARK SESSION + VOLUMES
# ============================================================

spark = SparkSession.builder \
    .appName("SpotifyMPD_Complete") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark.conf.set("spark.databricks.execution.timeout", "86400")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.outputs")

os.makedirs(RAW_VOL, exist_ok=True)
os.makedirs(OUT_VOL, exist_ok=True)

print("Spark ready:", spark.version)
print("Volumes ready.")

# ============================================================
# 3. KAGGLE DOWNLOAD
# ============================================================

def download_to_volume():
    print("\n--- KAGGLE DOWNLOAD ---")
    existing = [f for f in os.listdir(RAW_VOL) if f.endswith(".json")]
    if len(existing) >= 1000:
        print(f"MPD already in volume ({len(existing)} files) - skipped.")
        return

    subprocess.run(["pip", "install", "kaggle", "-q"], check=True)
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

    json_dir = "/tmp/mpd/data/"
    files_in_tmp = []
    if os.path.exists(json_dir):
        files_in_tmp = [f for f in os.listdir(json_dir) if f.endswith(".json")]

    if len(files_in_tmp) < 1000:
        print("Downloading from Kaggle to /tmp...")
        os.makedirs("/tmp/mpd/", exist_ok=True)
        import kaggle
        kaggle.api.authenticate()
        kaggle.api.dataset_download_files(
            "himanshuwagh/spotify-million",
            path="/tmp/mpd/",
            unzip=True,
            quiet=False
        )
        files_in_tmp = [f for f in os.listdir(json_dir) if f.endswith(".json")]

    print(f"{len(files_in_tmp)} files - copying to volume...")
    for i, filename in enumerate(files_in_tmp):
        shutil.copy2(f"{json_dir}{filename}", f"{RAW_VOL}{filename}")
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(files_in_tmp)} copied...")
    print("Done.")

# ============================================================
# 4. FLATTEN + CLEAN
# ============================================================

def load_and_flatten():
    print("\n--- FLATTEN + CLEAN ---")
    clean_path = OUT_VOL + "mpd_clean_flat/"

    if os.path.exists(clean_path):
        try:
            df = spark.read.parquet(clean_path)
            count = df.count()
            if count > 0:
                print(f"Checkpoint loaded: {count:,} rows")
                return df
        except Exception:
            pass

    print("Building from JSON files...")
    raw_df = spark.read.option("multiLine", "true").json(RAW_VOL)
    playlists_df = raw_df.select(F.explode("playlists").alias("pl"))

    playlists_flat = playlists_df.select(
        F.col("pl.pid").alias("playlist_id"),
        F.col("pl.name").alias("playlist_name"),
        F.col("pl.num_followers").alias("num_followers"),
        F.col("pl.num_tracks").alias("num_tracks"),
        F.col("pl.modified_at").alias("modified_at"),
        F.col("pl.duration_ms").alias("playlist_duration_ms"),
        F.col("pl.tracks").alias("tracks"),
    )

    tracks_df = playlists_flat.select(
        "playlist_id", "playlist_name", "num_followers",
        "num_tracks", "modified_at", "playlist_duration_ms",
        F.explode("tracks").alias("track")
    )

    flat_df = tracks_df.select(
        "playlist_id", "playlist_name", "num_followers",
        "num_tracks", "modified_at", "playlist_duration_ms",
        F.col("track.pos").alias("track_position"),
        F.col("track.track_name").alias("track_name"),
        F.col("track.track_uri").alias("track_uri"),
        F.col("track.artist_name").alias("artist_name"),
        F.col("track.artist_uri").alias("artist_uri"),
        F.col("track.album_name").alias("album_name"),
        F.col("track.album_uri").alias("album_uri"),
        F.col("track.duration_ms").alias("duration_ms"),
    )

    df_clean = flat_df.dropDuplicates()

    for col in ["playlist_name", "track_name", "track_uri",
                "artist_name", "artist_uri", "album_name", "album_uri"]:
        df_clean = df_clean.withColumn(col, F.trim(F.col(col)))

    df_clean = df_clean \
        .withColumn("track_id",
                    F.element_at(F.split("track_uri", ":"), -1)) \
        .withColumn("artist_id",
                    F.element_at(F.split("artist_uri", ":"), -1)) \
        .withColumn("album_id",
                    F.element_at(F.split("album_uri", ":"), -1)) \
        .withColumn("duration_min",
                    F.round(F.col("duration_ms") / 1000 / 60, 4)) \
        .withColumn("playlist_duration_min",
                    F.round(F.col("playlist_duration_ms") / 1000 / 60, 4)) \
        .withColumn("modified_at_datetime",
                    F.from_unixtime("modified_at").cast("timestamp"))

    total = df_clean.count()
    print(f"Total rows:       {total:,}")
    print(f"Unique playlists: {df_clean.select('playlist_id').distinct().count():,}")
    print(f"Unique tracks:    {df_clean.select('track_id').distinct().count():,}")
    print(f"Unique artists:   {df_clean.select('artist_id').distinct().count():,}")

    df_clean.write.mode("overwrite").parquet(clean_path)
    print("Checkpoint saved.")
    return df_clean

# ============================================================
# 5. COMPUTE POPULARITY (SparkSQL)
# ============================================================

def compute_popularity(df_clean):
    print("\n--- COMPUTE POPULARITY (SparkSQL) ---")

    pop_path = OUT_VOL + "track_popularity/"
    if os.path.exists(pop_path):
        try:
            track_pop = spark.read.parquet(pop_path)
            if track_pop.count() > 0:
                print("Checkpoint found — loading saved popularity, skipping rebuild.")
                return track_pop
        except Exception:
            pass

    df_clean.createOrReplaceTempView("tracks")

    track_pop = spark.sql("""
        SELECT
            track_id,
            track_name,
            artist_name,
            COUNT(DISTINCT playlist_id) AS playlist_count
        FROM tracks
        GROUP BY track_id, track_name, artist_name
        ORDER BY playlist_count DESC
    """)

    print("\nTop 10 Artists (SparkSQL):")
    spark.sql("""
        SELECT artist_name,
               COUNT(DISTINCT track_id)    AS unique_tracks,
               COUNT(DISTINCT playlist_id) AS playlist_appearances
        FROM tracks
        GROUP BY artist_name
        ORDER BY playlist_appearances DESC
        LIMIT 10
    """).show(truncate=False)

    print("\nTop 20 Tracks (SparkSQL):")
    spark.sql("""
        SELECT track_name, artist_name,
               COUNT(DISTINCT playlist_id) AS playlist_count
        FROM tracks
        GROUP BY track_name, artist_name, track_id
        ORDER BY playlist_count DESC
        LIMIT 20
    """).show(truncate=False)

    percentiles = track_pop.approxQuantile(
        "playlist_count", [HIGH_PERCENTILE, MID_PERCENTILE], 0.01
    )
    high_t = percentiles[0]
    mid_t  = percentiles[1]

    track_pop = track_pop.withColumn(
        "popularity_tier",
        F.when(F.col("playlist_count") >= high_t, "High")
         .when(F.col("playlist_count") >= mid_t,  "Medium")
         .otherwise("Low")
    )

    print(f"\nHigh (top 10%):  >= {high_t:.0f} playlists")
    print(f"Medium (70-90%): >= {mid_t:.0f} playlists")
    print(f"Low (long-tail): <  {mid_t:.0f} playlists")

    print("\nTier distribution:")
    track_pop.groupBy("popularity_tier") \
        .agg(F.count("*").alias("tracks"),
             F.avg("playlist_count").alias("avg_playlists"),
             F.max("playlist_count").alias("max_playlists")) \
        .orderBy(F.desc("avg_playlists")).show()

    track_pop.write.mode("overwrite").parquet(OUT_VOL + "track_popularity/")
    return track_pop

# ============================================================
# 6. MODELING TABLE
# ============================================================

def prepare_modeling(df_clean, track_pop):
    print("\n--- MODELING TABLE ---")

    modeling_path  = OUT_VOL + "mpd_modeling/"
    playlist_path  = OUT_VOL + "playlist_index/"
    track_path     = OUT_VOL + "track_index/"

    if os.path.exists(modeling_path) and os.path.exists(playlist_path) and os.path.exists(track_path):
        print("Checkpoint found — loading saved modeling table, skipping rebuild.")
        modeling_df    = spark.read.parquet(modeling_path)
        playlist_index = spark.read.parquet(playlist_path)
        track_index    = spark.read.parquet(track_path)
        modeling_df.createOrReplaceTempView("modeling")
        return modeling_df, playlist_index, track_index

    interactions = df_clean \
        .select("playlist_id", "track_id") \
        .dropDuplicates(["playlist_id", "track_id"]) \
        .join(track_pop.select("track_id", "playlist_count", "popularity_tier"),
              on="track_id", how="left")

    # Collect distinct IDs to driver and assign sequential integers there.
    # Avoids Window.orderBy() without partitionBy (global sort = single partition).
    playlist_rows = interactions.select("playlist_id").distinct().collect()
    playlist_index = spark.createDataFrame(
        [(row["playlist_id"], i) for i, row in enumerate(playlist_rows)],
        ["playlist_id", "playlist_idx"]
    )

    track_rows = interactions.select("track_id").distinct().collect()
    track_index = spark.createDataFrame(
        [(row["track_id"], i) for i, row in enumerate(track_rows)],
        ["track_id", "track_idx"]
    )

    modeling_df = interactions \
        .join(playlist_index, on="playlist_id", how="left") \
        .join(track_index,    on="track_id",    how="left") \
        .withColumn("confidence",
                    F.log1p(F.col("playlist_count").cast("float")))

    modeling_df.createOrReplaceTempView("modeling")
    print("\nModeling table statistics (SparkSQL):")
    spark.sql("""
        SELECT COUNT(*)                     AS total_interactions,
               COUNT(DISTINCT playlist_idx) AS unique_playlists,
               COUNT(DISTINCT track_idx)    AS unique_tracks,
               ROUND(AVG(confidence), 4)    AS avg_confidence,
               MAX(confidence)              AS max_confidence
        FROM modeling
    """).show()

    modeling_df.write.mode("overwrite").parquet(modeling_path)
    playlist_index.write.mode("overwrite").parquet(playlist_path)
    track_index.write.mode("overwrite").parquet(track_path)

    return modeling_df, playlist_index, track_index

# ============================================================
# 7. ALS TRAINING
# ============================================================

def train_als(modeling_df):
    print("\n--- ALS TRAINING ---")
    print(f"Rank: {ALS_RANK} | Iterations: {ALS_MAX_ITER} | RegParam: {ALS_REG_PARAM} | Alpha: {ALS_ALPHA}")

    train_df, test_df = modeling_df.randomSplit([0.8, 0.2], seed=ALS_SEED)
    print(f"Train: {train_df.count():,} | Test: {test_df.count():,}")

    model_path = OUT_VOL + "als_model"
    if FORCE_RETRAIN and os.path.exists(model_path):
        shutil.rmtree(model_path)
        print("FORCE_RETRAIN=True — deleted old model, retraining with improved params.")
    if os.path.exists(model_path):
        print("Checkpoint found — loading saved model, skipping training.")
        model = ALSModel.load(model_path)
        return model, train_df, test_df

    als = ALS(
        userCol="playlist_idx",
        itemCol="track_idx",
        ratingCol="confidence",
        rank=ALS_RANK,
        maxIter=ALS_MAX_ITER,
        regParam=ALS_REG_PARAM,
        alpha=ALS_ALPHA,
        implicitPrefs=True,
        coldStartStrategy="drop",
        seed=ALS_SEED
    )

    print("Training...")
    model = als.fit(train_df)
    model.write().overwrite().save(model_path)
    print("Model saved.")
    return model, train_df, test_df

# ============================================================
# 8. EVALUATION
# ============================================================

def evaluate(model, test_df, track_pop, track_index):
    print(f"\n--- EVALUATION @K={K} ---")

    # RMSE
    preds = model.transform(test_df.fillna(0, subset=["confidence"]))
    rmse  = RegressionEvaluator(
        metricName="rmse",
        labelCol="confidence",
        predictionCol="prediction"
    ).evaluate(preds.dropna(subset=["prediction"]))
    print(f"RMSE: {rmse:.4f}")

    # Driver-side top-K: collect 2000 user factors, compute scores with numpy.
    # Fast (~30s), no Arrow/UDF overhead, correct for small EVAL_SAMPLE.
    print(f"Generating recommendations (driver-side numpy, {EVAL_SAMPLE} users, {CANDIDATE_K} candidates)...")
    import numpy as np
    import pandas as pd

    test_user_ids = test_df.select("playlist_idx").distinct().limit(EVAL_SAMPLE)
    print(f"Evaluating on {EVAL_SAMPLE:,} sampled test users.")

    # Collect user factors to driver
    user_factors_pd = model.userFactors.withColumnRenamed("id", "playlist_idx") \
        .join(test_user_ids, on="playlist_idx", how="inner") \
        .toPandas()
    user_ids = user_factors_pd["playlist_idx"].values
    user_mat  = np.array(user_factors_pd["features"].tolist(), dtype=np.float32)

    # Collect item factors + tier info to driver
    item_factors_pd = model.itemFactors.toPandas()
    item_ids = item_factors_pd["id"].values
    item_mat  = np.array(item_factors_pd["features"].tolist(), dtype=np.float32)

    # Build tier masks for forced diversity
    tier_info = track_index.join(
        track_pop.select("track_id", "popularity_tier"), on="track_id", how="left"
    ).select("track_idx", "popularity_tier").toPandas()
    tier_map = dict(zip(tier_info["track_idx"], tier_info["popularity_tier"]))
    item_tiers = np.array([tier_map.get(int(iid), "High") for iid in item_ids])
    high_mask   = np.where(item_tiers == "High")[0]
    medium_mask = np.where(item_tiers == "Medium")[0]
    low_mask    = np.where(item_tiers == "Low")[0]
    print(f"Tier counts — High: {len(high_mask):,}, Medium: {len(medium_mask):,}, Low: {len(low_mask):,}")

    # Batch matrix multiply: (n_users x rank) @ (rank x n_items) = (n_users x n_items)
    print("Computing scores...")
    scores_mat = user_mat @ item_mat.T

    # Forced diversity candidates: 60 High + 20 Medium + 20 Low per user
    N_HIGH_C = 60
    N_MED_C  = 20
    N_LOW_C  = 20
    rows = []
    for i, uid in enumerate(user_ids):
        scores = scores_mat[i]
        # Top High
        top_high = high_mask[np.argpartition(scores[high_mask], -N_HIGH_C)[-N_HIGH_C:]]
        # Top Medium
        n_med = min(N_MED_C, len(medium_mask))
        top_med = medium_mask[np.argpartition(scores[medium_mask], -n_med)[-n_med:]]
        # Top Low
        n_low = min(N_LOW_C, len(low_mask))
        top_low = low_mask[np.argpartition(scores[low_mask], -n_low)[-n_low:]]
        # Combine
        combined = list(dict.fromkeys(
            list(top_high) + list(top_med) + list(top_low)
        ))[:CANDIDATE_K]
        for rank, idx in enumerate(combined):
            rows.append((int(uid), int(item_ids[idx]), float(scores[idx]), rank))

    rec_pd = pd.DataFrame(rows, columns=["playlist_idx", "track_idx", "score", "rank"])
    recommendations_flat = spark.createDataFrame(rec_pd)
    recommendations_flat.write.mode("overwrite").parquet(OUT_VOL + "recommendations_raw/")
    recommendations_flat = spark.read.parquet(OUT_VOL + "recommendations_raw/")
    print("Recommendations ready.")

    # Ground truth
    test_tracks = test_df.groupBy("playlist_idx") \
        .agg(F.collect_set("track_idx").alias("actual"))

    # Rebuild ordered arrays for metric UDFs — only top-K by ALS score
    recommended_arrays = recommendations_flat \
        .filter(F.col("rank") < K) \
        .orderBy("playlist_idx", "rank") \
        .groupBy("playlist_idx") \
        .agg(F.collect_list("track_idx").alias("recommended"))

    eval_df = recommended_arrays.join(test_tracks, on="playlist_idx", how="inner")

    # Metric UDFs - pure Python, no higher-order functions
    def precision_k(rec, act):
        if not rec or not act: return 0.0
        return len(set(rec[:K]) & set(act)) / K

    def recall_k(rec, act):
        if not act: return 0.0
        return len(set(rec[:K]) & set(act)) / len(act)

    def ndcg_k(rec, act):
        if not act or not rec: return 0.0
        dcg  = sum(1/math.log2(i+2) for i,x in
                   enumerate(rec[:K]) if x in set(act))
        idcg = sum(1/math.log2(i+2) for i in range(min(K, len(act))))
        return dcg/idcg if idcg > 0 else 0.0

    p_udf = F.udf(lambda r,a: float(precision_k(r,a)), FloatType())
    r_udf = F.udf(lambda r,a: float(recall_k(r,a)),    FloatType())
    n_udf = F.udf(lambda r,a: float(ndcg_k(r,a)),      FloatType())

    m = eval_df \
        .withColumn("p", p_udf("recommended", "actual")) \
        .withColumn("r", r_udf("recommended", "actual")) \
        .withColumn("n", n_udf("recommended", "actual")) \
        .agg(F.avg("p").alias("precision"),
             F.avg("r").alias("recall"),
             F.avg("n").alias("ndcg")) \
        .collect()[0]

    print(f"Precision@{K}: {m['precision']:.4f}")
    print(f"Recall@{K}:    {m['recall']:.4f}")
    print(f"NDCG@{K}:      {m['ndcg']:.4f}")

    # Long-tail coverage — only over the top-K ALS recommendations
    all_rec = recommendations_flat.filter(F.col("rank") < K).select("track_idx")
    low_idxs = track_pop.filter(F.col("popularity_tier") == "Low") \
        .join(track_index, on="track_id", how="inner") \
        .select("track_idx")
    total   = all_rec.count()
    lt_hits = all_rec.join(low_idxs, on="track_idx", how="inner").count()
    lt_cov  = lt_hits / total * 100
    print(f"Long-tail Coverage: {lt_cov:.1f}%")

    # Popularity baseline - Unity Catalog compatible
    top_k = track_pop.orderBy(F.desc("playlist_count")) \
        .limit(K) \
        .join(track_index, on="track_id", how="inner") \
        .select("track_idx").collect()
    top_k_idx = set([r["track_idx"] for r in top_k])

    top_k_df = spark.createDataFrame(
        [(i,) for i in top_k_idx], ["track_idx"]
    )
    baseline_hits = test_tracks \
        .withColumn("track_idx", F.explode("actual")) \
        .join(top_k_df.hint("broadcast"), on="track_idx", how="inner") \
        .select("playlist_idx").distinct().count()
    baseline = baseline_hits / test_tracks.count()
    print(f"Baseline Precision@{K}: {baseline:.4f}")

    return {
        "rmse":        rmse,
        "precision":   m["precision"],
        "recall":      m["recall"],
        "ndcg":        m["ndcg"],
        "lt_coverage": lt_cov,
        "baseline":    baseline
    }, recommendations_flat

# ============================================================
# 9. LONG-TAIL RE-RANKING
# ============================================================

def longtail_reranking(recommendations_flat, track_index, track_pop, test_df):
    print("\n--- LONG-TAIL RE-RANKING ---")

    N_HIGH   = 7  # slots for High tier
    N_MEDIUM = 1  # slots for Medium tier
    N_LOW    = 2  # slots for Low tier (counted in Long-tail Coverage)

    track_tiers = track_index.join(
        track_pop.select("track_id", "popularity_tier"),
        on="track_id", how="left"
    )

    recs_with_tier = recommendations_flat.join(
        track_tiers.select("track_idx", "popularity_tier"),
        on="track_idx", how="left"
    )

    w_score = Window.partitionBy("playlist_idx").orderBy(F.desc("score"))

    high_slots = recs_with_tier \
        .filter(F.col("popularity_tier") == "High") \
        .withColumn("slot_rank", F.rank().over(w_score)) \
        .filter(F.col("slot_rank") <= N_HIGH) \
        .withColumn("new_rank", F.col("slot_rank"))

    medium_slots = recs_with_tier \
        .filter(F.col("popularity_tier") == "Medium") \
        .withColumn("slot_rank", F.rank().over(w_score)) \
        .filter(F.col("slot_rank") <= N_MEDIUM) \
        .withColumn("new_rank", F.col("slot_rank") + N_HIGH)

    low_slots = recs_with_tier \
        .filter(F.col("popularity_tier") == "Low") \
        .withColumn("slot_rank", F.rank().over(w_score)) \
        .filter(F.col("slot_rank") <= N_LOW) \
        .withColumn("new_rank", F.col("slot_rank") + N_HIGH + N_MEDIUM)

    recs_reranked = high_slots \
        .unionByName(medium_slots, allowMissingColumns=True) \
        .unionByName(low_slots, allowMissingColumns=True)

    recs_reranked.write.mode("overwrite").parquet(
        OUT_VOL + "recommendations_reranked/")
    recs_reranked = spark.read.parquet(OUT_VOL + "recommendations_reranked/")

    total = recs_reranked.count()
    lt    = recs_reranked.filter(F.col("popularity_tier") == "Low").count()
    lt_cov_reranked = lt / total * 100
    print(f"Long-tail Coverage after re-ranking: {lt_cov_reranked:.1f}%")

    recs_reranked.createOrReplaceTempView("recommendations_reranked")
    print("\nTier distribution (SparkSQL):")
    spark.sql("""
        SELECT popularity_tier,
               COUNT(*) AS count,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS percentage
        FROM recommendations_reranked
        GROUP BY popularity_tier
        ORDER BY count DESC
    """).show()

    # Post-reranking Precision/Recall/NDCG
    test_tracks = test_df.groupBy("playlist_idx") \
        .agg(F.collect_set("track_idx").alias("actual"))

    reranked_arrays = recs_reranked \
        .orderBy("playlist_idx", "new_rank") \
        .groupBy("playlist_idx") \
        .agg(F.collect_list("track_idx").alias("recommended"))

    eval_reranked = reranked_arrays.join(test_tracks, on="playlist_idx", how="inner")

    def precision_k(rec, act):
        if not rec or not act: return 0.0
        return len(set(rec[:K]) & set(act)) / K

    def recall_k(rec, act):
        if not act: return 0.0
        return len(set(rec[:K]) & set(act)) / len(act)

    def ndcg_k(rec, act):
        if not act or not rec: return 0.0
        dcg  = sum(1/math.log2(i+2) for i,x in enumerate(rec[:K]) if x in set(act))
        idcg = sum(1/math.log2(i+2) for i in range(min(K, len(act))))
        return dcg/idcg if idcg > 0 else 0.0

    p_udf = F.udf(lambda r,a: float(precision_k(r,a)), FloatType())
    r_udf = F.udf(lambda r,a: float(recall_k(r,a)),    FloatType())
    n_udf = F.udf(lambda r,a: float(ndcg_k(r,a)),      FloatType())

    m = eval_reranked \
        .withColumn("p", p_udf("recommended", "actual")) \
        .withColumn("r", r_udf("recommended", "actual")) \
        .withColumn("n", n_udf("recommended", "actual")) \
        .agg(F.avg("p").alias("precision"),
             F.avg("r").alias("recall"),
             F.avg("n").alias("ndcg")) \
        .collect()[0]

    print(f"\nPost-reranking Precision@{K}: {m['precision']:.4f}")
    print(f"Post-reranking Recall@{K}:    {m['recall']:.4f}")
    print(f"Post-reranking NDCG@{K}:      {m['ndcg']:.4f}")

    print("Saved.")
    return recs_reranked, {
        "precision_reranked": m["precision"],
        "recall_reranked":    m["recall"],
        "ndcg_reranked":      m["ndcg"],
        "lt_coverage_reranked": lt_cov_reranked
    }

# ============================================================
# 10. PRINT RESULTS
# ============================================================

def print_results(results, reranked_results):
    print("\n" + "="*60)
    print("FINAL RESULTS")
    print("="*60)
    print(f"RMSE:                         {results['rmse']:.4f}")
    print(f"")
    print(f"{'Metric':<30} {'ALS only':>10} {'+ Re-rank':>10}")
    print(f"{'-'*50}")
    print(f"{'Precision@'+str(K):<30} {results['precision']:>10.4f} {reranked_results['precision_reranked']:>10.4f}")
    print(f"{'Recall@'+str(K):<30} {results['recall']:>10.4f} {reranked_results['recall_reranked']:>10.4f}")
    print(f"{'NDCG@'+str(K):<30} {results['ndcg']:>10.4f} {reranked_results['ndcg_reranked']:>10.4f}")
    print(f"{'Long-tail Coverage':<30} {results['lt_coverage']:>9.1f}% {reranked_results['lt_coverage_reranked']:>9.1f}%")
    print(f"{'-'*50}")
    print(f"Baseline Precision@{K}:         {results['baseline']:.4f}")
    print("="*60)
    print(f"\nAll outputs: {OUT_VOL}")
    print(f"  als_model/                - trained ALS (Rank {ALS_RANK})")
    print("  recommendations_raw/      - raw ALS output")
    print("  recommendations_reranked/ - final re-ranked recommendations")

# ============================================================
# 11. MAIN
# ============================================================

def main():
    import time
    pipeline_start = time.time()

    # Batch processing: all stages run on schedule over stored data.
    # Chosen over streaming because playlist co-occurrence patterns
    # are stable and do not require sub-minute latency.

    t0 = time.time()
    download_to_volume()
    print(f"[TIMING] Download:    {time.time()-t0:.1f}s")

    t0 = time.time()
    df_clean = load_and_flatten()
    print(f"[TIMING] Flatten:     {time.time()-t0:.1f}s")
    print("\n[EXECUTION PLAN] Flatten DataFrame:")
    df_clean.explain()

    t0 = time.time()
    track_pop = compute_popularity(df_clean)
    print(f"[TIMING] Popularity:  {time.time()-t0:.1f}s")

    t0 = time.time()
    modeling_df, playlist_index, track_index = prepare_modeling(df_clean, track_pop)
    print(f"[TIMING] Modeling:    {time.time()-t0:.1f}s")
    print("\n[EXECUTION PLAN] Modeling DataFrame:")
    modeling_df.explain()

    t0 = time.time()
    model, train_df, test_df = train_als(modeling_df)
    print(f"[TIMING] ALS Train:   {time.time()-t0:.1f}s")

    t0 = time.time()
    results, recommendations = evaluate(model, test_df, track_pop, track_index)
    print(f"[TIMING] Evaluation:  {time.time()-t0:.1f}s")

    t0 = time.time()
    recs_reranked, reranked_results = longtail_reranking(recommendations, track_index, track_pop, test_df)
    print(f"[TIMING] Re-Ranking:  {time.time()-t0:.1f}s")

    print_results(results, reranked_results)
    print(f"\n[TIMING] Total pipeline: {(time.time()-pipeline_start)/60:.1f} min")
    print("\nPipeline complete.")

main()


Spark ready: 4.1.0
Volumes ready.

--- KAGGLE DOWNLOAD ---
MPD already in volume (1000 files) - skipped.
[TIMING] Download:    0.4s

--- FLATTEN + CLEAN ---
Checkpoint loaded: 66,346,428 rows
[TIMING] Flatten:     1.3s

[EXECUTION PLAN] Flatten DataFrame:
== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonScan parquet [playlist_id#13594L,playlist_name#13595,num_followers#13596L,num_tracks#13597L,modified_at#13598L,playlist_duration_ms#13599L,track_position#13600L,track_name#13601,track_uri#13602,artist_name#13603,artist_uri#13604,album_name#13605,album_uri#13606,duration_ms#13607L,track_id#13608,artist_id#13609,album_id#13610,duration_min#13611,playlist_duration_min#13612,modified_at_datetime#13613] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/spotify_project/mpd/outputs/mpd_clean_flat], OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<playlist_id:bigint,playlist_name:string,num_follow

In [0]:
 """
Export Pipeline Outputs for Streamlit Dashboard
================================================
Run this as a NEW cell in Databricks AFTER the main pipeline completes.
Then download the CSVs from the Volume to your local machine.
"""

import pandas as pd
from pyspark.sql import functions as F

CATALOG = "spotify_project"
SCHEMA  = "mpd"
OUT_VOL = f"/Volumes/{CATALOG}/{SCHEMA}/outputs/"

print("Exporting data for Streamlit dashboard...")

# 1. Top 50 Tracks
track_pop = spark.read.parquet(OUT_VOL + "track_popularity/")
top_tracks = track_pop.orderBy(F.desc("playlist_count")).limit(50).toPandas()
top_tracks.to_csv(OUT_VOL + "export_top_tracks.csv", index=False)
print("✓ Top tracks exported")

# 2. Tier Distribution
tier_dist = track_pop.groupBy("popularity_tier") \
    .agg(F.count("*").alias("track_count")) \
    .toPandas()
tier_dist.to_csv(OUT_VOL + "export_tier_distribution.csv", index=False)
print("✓ Tier distribution exported")

# 3. Track info (idx → name, artist, tier)
track_index = spark.read.parquet(OUT_VOL + "track_index/")
track_info = track_index.join(
    track_pop.select("track_id", "track_name", "artist_name", "playlist_count", "popularity_tier"),
    on="track_id", how="left"
).toPandas()
track_info.to_csv(OUT_VOL + "export_track_info.csv", index=False)
print("✓ Track info exported")

# 4. Recommendations (reranked, top-10 per user)
recs = spark.read.parquet(OUT_VOL + "recommendations_reranked/")
recs_pd = recs.select("playlist_idx", "track_idx", "score", "new_rank", "popularity_tier") \
    .toPandas()
recs_pd.to_csv(OUT_VOL + "export_recommendations.csv", index=False)
print("✓ Recommendations exported")

# 5. Playlist index (sample of 500 for dropdown)
playlist_index = spark.read.parquet(OUT_VOL + "playlist_index/")
playlist_sample = playlist_index.limit(500).toPandas()
playlist_sample.to_csv(OUT_VOL + "export_playlist_index.csv", index=False)
print("✓ Playlist index exported")

# 6. Original tracks per playlist (same playlists used in recommendations)
recs_playlists = spark.read.parquet(OUT_VOL + "recommendations_reranked/") \
    .select("playlist_idx").distinct()

modeling = spark.read.parquet(OUT_VOL + "mpd_modeling/")
track_index = spark.read.parquet(OUT_VOL + "track_index/")
track_pop = spark.read.parquet(OUT_VOL + "track_popularity/")

modeling_slim = modeling.select("playlist_idx", "track_idx")
track_index_slim = track_index.select("track_idx", "track_id")
track_pop_slim = track_pop.select("track_id", "track_name", "artist_name", "popularity_tier")

original_tracks = modeling_slim \
    .join(recs_playlists, on="playlist_idx", how="inner") \
    .join(track_index_slim, on="track_idx", how="left") \
    .join(track_pop_slim, on="track_id", how="left") \
    .select("playlist_idx", "track_idx", "track_name", "artist_name", "popularity_tier") \
    .toPandas()
original_tracks.to_csv(OUT_VOL + "export_original_tracks.csv", index=False)
print("✓ Original playlist tracks exported")

print(f"\nAll exports saved to: {OUT_VOL}")
print("Download them via Databricks UI: Catalog → Volumes → outputs → download each export_*.csv")


Exporting data for Streamlit dashboard...
✓ Top tracks exported
✓ Tier distribution exported
✓ Track info exported
✓ Recommendations exported
✓ Playlist index exported
✓ Original playlist tracks exported

All exports saved to: /Volumes/spotify_project/mpd/outputs/
Download them via Databricks UI: Catalog → Volumes → outputs → download each export_*.csv


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 123.9 MB/s eta 0:00:00
  Created wheel for thrift: filename=thrift-0.22.0-cp312-cp312-linux_x86_64.whl size=411005 sha256=960c1a1467b6620402f18ab2eb86a09490cd4ee998ca5ef9fa6ccf7df85c25dd
  Stored in directory: /home/spark-8f3354f6-8d22-48c3-9b75-ab/.cache/pip/wheels/03/30/a2/6e53de46f9d8f098a6b7c4a57dbdcafa5f11e87274e139f50b
Successfully built thrift
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


2026-05-03 15:22:33.306 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6732522503367889>, line 20
     12 import os
     14 st.set_page_config(
     15     page_title="Spotify MPD Recommendation Pipeline",
     16     page_icon="🎵",
     17     layout="wide"
     18 )
---> 20 DATA_DIR = os.path.dirname(__file__)
     22 CATALOG = "spotify_project"
     23 SCHEMA  = "mpd"

NameError: name '__file__' is not defined